# 🏦 Loan Approval Prediction
### Supervised ML Pipeline | Preprocessing · Imbalance Handling · Model Comparison

**Goal:** Predict whether a loan application will be approved (`Loan_Status = Y/N`) using borrower features.


In [ ]:
# ── 1. Install & Import ─────────────────────────────────────────────────
!pip install imbalanced-learn scikit-learn xgboost lightgbm shap --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              f1_score, precision_score, recall_score)
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb
import lightgbm as lgb
import shap

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("✅ All libraries imported successfully.")


## 📂 Step 1 — Load Data

In [ ]:
from google.colab import files
uploaded = files.upload()           # ← choose loan_prediction.csv when prompted

import io
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()


## 🔍 Step 2 — Exploratory Data Analysis

In [ ]:
print("=" * 55)
print("BASIC INFO")
print("=" * 55)
df.info()

print("\n" + "=" * 55)
print("MISSING VALUES")
print("=" * 55)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Pct (%)': missing_pct})
print(missing_df[missing_df['Count'] > 0])

print("\n" + "=" * 55)
print("TARGET CLASS DISTRIBUTION")
print("=" * 55)
print(df['Loan_Status'].value_counts())
print(df['Loan_Status'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Loan Approval — Exploratory Analysis', fontsize=15, fontweight='bold')

# Class distribution
ax = axes[0, 0]
counts = df['Loan_Status'].value_counts()
bars = ax.bar(['Approved (Y)', 'Rejected (N)'], counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(val), ha='center', fontweight='bold')
ax.set_title('Class Distribution'); ax.set_ylabel('Count')

# Applicant Income by Loan Status
ax = axes[0, 1]
df.boxplot(column='ApplicantIncome', by='Loan_Status', ax=ax,
           boxprops=dict(color='#2980b9'), medianprops=dict(color='red', linewidth=2))
ax.set_title('Applicant Income by Loan Status')
ax.set_xlabel('Loan Status'); ax.set_ylabel('Income')
plt.sca(ax); plt.title('Applicant Income vs Loan Status')

# Loan Amount distribution
ax = axes[0, 2]
df['LoanAmount'].dropna().hist(bins=30, ax=ax, color='#9b59b6', edgecolor='white', alpha=0.8)
ax.set_title('Loan Amount Distribution'); ax.set_xlabel('Loan Amount (K)')

# Credit History
ax = axes[1, 0]
ct = pd.crosstab(df['Credit_History'], df['Loan_Status'], normalize='index') * 100
ct.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='white', rot=0)
ax.set_title('Approval Rate by Credit History')
ax.set_xlabel('Credit History (0=Bad, 1=Good)'); ax.set_ylabel('Approval %')
ax.legend(['Rejected', 'Approved'])

# Property Area
ax = axes[1, 1]
ct2 = pd.crosstab(df['Property_Area'], df['Loan_Status'])
ct2.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='white', rot=0)
ax.set_title('Approvals by Property Area')
ax.set_xlabel('Property Area'); ax.set_ylabel('Count')
ax.legend(['Rejected', 'Approved'])

# Education
ax = axes[1, 2]
ct3 = pd.crosstab(df['Education'], df['Loan_Status'], normalize='index') * 100
ct3.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='white', rot=0)
ax.set_title('Approval Rate by Education')
ax.set_xlabel('Education'); ax.set_ylabel('Approval %')
ax.legend(['Rejected', 'Approved'])

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved.")


## 🛠 Step 3 — Preprocessing

In [ ]:
# ── Drop ID column
df_clean = df.drop(columns=['Loan_ID'])

# ── Feature Engineering: Total Income
df_clean['TotalIncome'] = df_clean['ApplicantIncome'] + df_clean['CoapplicantIncome']
df_clean['TotalIncome_log'] = np.log1p(df_clean['TotalIncome'])
df_clean['LoanAmount_log'] = np.log1p(df_clean['LoanAmount'])
df_clean['EMI'] = df_clean['LoanAmount'] / df_clean['Loan_Amount_Term']
df_clean['Balance_Income'] = df_clean['TotalIncome'] - (df_clean['EMI'] * 1000)

# ── Encode target
df_clean['Loan_Status'] = (df_clean['Loan_Status'] == 'Y').astype(int)

# ── Separate features / target
X = df_clean.drop(columns=['Loan_Status', 'ApplicantIncome', 'CoapplicantIncome', 'TotalIncome'])
y = df_clean['Loan_Status']

# ── Identify column types
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

print("Categorical:", cat_cols)
print("Numerical  :", num_cols)

# ── Impute missing values
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

# ── Label-encode categoricals
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

print(f"\nFinal feature set shape: {X.shape}")
print(f"Class balance (0=Rejected, 1=Approved):\n{y.value_counts()}")


## ✂️ Step 4 — Train / Test Split & Imbalance Handling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train class dist:\n{y_train.value_counts()}")
print(f"\nTest  class dist:\n{y_test.value_counts()}")

# ── Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── SMOTE on training set
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)
print(f"\nAfter SMOTE — Train class dist:\n{pd.Series(y_train_sm).value_counts()}")


## 🤖 Step 5 — Train & Evaluate All Models

In [ ]:
models = {
    'Logistic Regression'      : LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree'            : DecisionTreeClassifier(random_state=SEED, max_depth=6),
    'Random Forest'            : RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight='balanced'),
    'Gradient Boosting'        : GradientBoostingClassifier(n_estimators=200, random_state=SEED),
    'XGBoost'                  : xgb.XGBClassifier(n_estimators=200, random_state=SEED,
                                                    eval_metric='logloss', verbosity=0),
    'LightGBM'                 : lgb.LGBMClassifier(n_estimators=200, random_state=SEED, verbose=-1),
}

results = {}

for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]

    results[name] = {
        'Precision' : precision_score(y_test, y_pred),
        'Recall'    : recall_score(y_test, y_pred),
        'F1'        : f1_score(y_test, y_pred),
        'ROC-AUC'   : roc_auc_score(y_test, y_proba),
        'y_pred'    : y_pred,
        'y_proba'   : y_proba,
        'model'     : model
    }
    print(f"✔ {name:28s} | F1={results[name]['F1']:.3f} | AUC={results[name]['ROC-AUC']:.3f}")

print("\n✅ All models trained.")


## 📊 Step 6 — Metrics Comparison Table

In [ ]:
metrics_df = pd.DataFrame({
    name: {k: v for k, v in vals.items() if k in ['Precision','Recall','F1','ROC-AUC']}
    for name, vals in results.items()
}).T.round(4)

metrics_df = metrics_df.sort_values('ROC-AUC', ascending=False)
print(metrics_df.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(metrics_df.astype(float), annot=True, fmt='.3f', cmap='YlGn',
            linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0)
ax.set_title('Model Metrics Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('metrics_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


## 📈 Step 7 — ROC & Precision-Recall Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']

for (name, vals), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, vals['y_proba'])
    ax1.plot(fpr, tpr, color=color, lw=2,
             label=f"{name} (AUC={vals['ROC-AUC']:.3f})")

    prec, rec, _ = precision_recall_curve(y_test, vals['y_proba'])
    ax2.plot(rec, prec, color=color, lw=2, label=name)

ax1.plot([0,1],[0,1],'k--', lw=1)
ax1.set_title('ROC Curves', fontsize=12, fontweight='bold')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.legend(fontsize=7.5); ax1.grid(alpha=0.3)

ax2.set_title('Precision–Recall Curves', fontsize=12, fontweight='bold')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.legend(fontsize=7.5); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=120, bbox_inches='tight')
plt.show()


## 🏆 Step 8 — Best Model Deep-Dive

In [ ]:
best_name = metrics_df['ROC-AUC'].idxmax()
best = results[best_name]
print(f"Best Model: {best_name}\n")
print(classification_report(y_test, best['y_pred'],
      target_names=['Rejected (0)', 'Approved (1)']))

# Confusion matrix
cm = confusion_matrix(y_test, best['y_pred'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Rejected','Approved'],
            yticklabels=['Rejected','Approved'])
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


## 🔮 Step 9 — SHAP Feature Importance (Best Model)

In [ ]:
import shap

best_model = best['model']

# Use TreeExplainer for tree-based; LinearExplainer for Logistic Regression
try:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test_sc)
    # For multi-output SHAP (some models return list), take class=1
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values
except Exception:
    explainer = shap.LinearExplainer(best_model, X_train_sm)
    shap_values = explainer.shap_values(X_test_sc)
    shap_vals = shap_values

plt.figure()
shap.summary_plot(shap_vals, X_test_sc,
                  feature_names=X.columns.tolist(),
                  plot_type='bar', show=False,
                  plot_size=(9, 5))
plt.title(f'Feature Importance (SHAP) — {best_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()

plt.figure()
shap.summary_plot(shap_vals, X_test_sc,
                  feature_names=X.columns.tolist(), show=False,
                  plot_size=(9, 6))
plt.title(f'SHAP Beeswarm — {best_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()


## ⚖️ Step 10 — Decision Threshold Analysis

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores, prec_scores, rec_scores = [], [], []

for t in thresholds:
    y_pred_t = (best['y_proba'] >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))
    prec_scores.append(precision_score(y_test, y_pred_t, zero_division=0))
    rec_scores.append(recall_score(y_test, y_pred_t, zero_division=0))

optimal_threshold = thresholds[np.argmax(f1_scores)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, f1_scores,   'b-o', ms=4, label='F1-Score')
ax.plot(thresholds, prec_scores, 'g-s', ms=4, label='Precision')
ax.plot(thresholds, rec_scores,  'r-^', ms=4, label='Recall')
ax.axvline(x=optimal_threshold, color='orange', linestyle='--', lw=2,
           label=f'Optimal Threshold = {optimal_threshold:.2f}')
ax.set_title('Threshold vs Metrics', fontweight='bold')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\n📌 Optimal threshold (max F1): {optimal_threshold:.2f}")
print(f"   At this threshold:")
y_opt = (best['y_proba'] >= optimal_threshold).astype(int)
print(f"   Precision = {precision_score(y_test, y_opt):.3f}")
print(f"   Recall    = {recall_score(y_test, y_opt):.3f}")
print(f"   F1        = {f1_score(y_test, y_opt):.3f}")


## 💼 Step 11 — Business Interpretation

### Key Insights from SHAP Analysis

| Feature | Business Meaning |
|---|---|
| **Credit History** | Strongest predictor — applicants with good credit (=1) are far more likely to be approved. Absence of credit history (=0) is the top rejection signal. |
| **TotalIncome_log** | Higher combined income (applicant + co-applicant) strongly increases approval probability. |
| **LoanAmount_log** | Smaller loan amounts relative to income improve approval odds. |
| **EMI** | High estimated monthly instalment relative to income is a risk flag. |
| **Property Area** | Semi-urban applicants tend to have higher approval rates than Rural. |
| **Dependents** | More dependents slightly increases risk perception. |

### Recommended Deployment Threshold
Use the **optimal threshold from Step 10** rather than 0.5.  
- **Lower threshold** (e.g. 0.35) → higher recall → approve more borderline applicants → accept more risk of default but capture more business.  
- **Higher threshold** (e.g. 0.55) → higher precision → stricter approvals → fewer defaults but potentially miss creditworthy applicants.  
- For a **risk-averse lender**, use the threshold that maximises **Precision** for the Rejected class.  
- For **growth-focused lending**, use the threshold that maximises **Recall** for the Approved class.

### Model Recommendation
Tree-based ensemble models (Random Forest / XGBoost / LightGBM) consistently outperform Logistic Regression on this dataset due to non-linear interactions (e.g. Credit History × Income).  
**XGBoost or LightGBM** are recommended for production given their superior AUC and interpretability via SHAP.


## 💾 Step 12 — Export Results

In [ ]:
# Save metrics CSV
metrics_df.to_csv('model_metrics.csv')
print("✅ model_metrics.csv saved")

# Save test predictions
pred_df = pd.DataFrame({
    'Actual'          : y_test.values,
    'Predicted'       : best['y_pred'],
    'Proba_Approved'  : best['y_proba']
})
pred_df.to_csv('test_predictions.csv', index=False)
print("✅ test_predictions.csv saved")

# Download all files
from google.colab import files
for fname in ['model_metrics.csv', 'test_predictions.csv',
              'eda_plots.png', 'metrics_heatmap.png',
              'roc_pr_curves.png', 'confusion_matrix.png',
              'shap_importance.png', 'threshold_analysis.png']:
    try:
        files.download(fname)
    except Exception:
        pass

print("\n🏁 Pipeline complete!")
print(f"Best model: {best_name}")
print(metrics_df.loc[best_name].to_string())
